# Higher Education: UC, CSU and Community Colleges
California's public higher education has three systems: the **University of California** (10 campuses plus UC College of the Law, SF, and UC Health's hospitals and clinics), the **California State University** (23 campuses) and the **California Community Colleges** (about 115 colleges in 72 districts). This page covers their finances: what they spend, what it is spent on, and who pays. The source is the U.S. Department of Education's IPEDS finance survey, filed by every public institution (`scripts/fetch_ipeds.py`). Fiscal year is the year the fiscal year ends (2022-23 is 2023).

**Read this first**
- **Scope:** each system's *whole* budget, not just state tax dollars. That means tuition, federal research grants, hospital revenue, housing and dining, gifts and local property taxes as well as state money. The state-money part is on the **revenue** charts ("State appropriations"). UC's hospitals and clinics are about a third of its spending, which is why UC dwarfs the other two.
- **Years:** FY2004-2023 for UC and CSU, FY2005-2023 for community colleges (only 77 of about 110 colleges filed in the survey's 2004 format, so that year is left out). The survey lags by a couple of years; FY2024 is not yet published.
- **Spending by function breaks twice.** Through FY2009 depreciation and interest are their own lines, which is why a "Depreciation, interest and other" band appears; from FY2010 they are spread across the functions. Plant operation and maintenance is its own line through FY2015 and is spread across the functions after. Dotted lines mark the breaks (FY2010 and FY2016).
- **'Other' grows after FY2016.** The filings' catch-all 'other expenses' line is near zero through FY2016 and then reaches 10 to 17% of spending in each system by 2023, so function shares after FY2016 are partly unexplained. It probably holds pension and retiree-health expense that institutions did not assign to a function, but the filings do not say.
- **Benefits include accounting entries.** Reported employee benefits include pension and retiree-health expense as calculated under accounting rules (GASB 68 and 75), which can swing a lot from year to year without any cash changing hands. Read benefit trends over several years.
- **Students:** full-time-equivalent (FTE) students come from IPEDS' 12-month instructional-activity survey and count credit instruction only, so they understate community college students (non-credit courses are not counted) and are a rough denominator. Per-student figures are for trends within a system, not for comparing systems.
- **Community colleges are not in the state-government pages or the local ones.** They are funded by a mix of state money and local property taxes, shown here as separate revenue sources. K-12 districts are on the school pages.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, line_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
ex = pd.read_csv(PROCESSED / "highered_expenses.csv")
nat = pd.read_csv(PROCESSED / "highered_natural.csv")
rev = pd.read_csv(PROCESSED / "highered_revenue.csv")
fte = pd.read_csv(PROCESSED / "highered_fte.csv")
from ca_taxes.highered import SYSTEMS, FUNCTION_ORDER, CORE, REVENUE_ORDER
HE = "Source: U.S. Dept. of Education, IPEDS finance survey (public institutions)"
BRK = {2009.5: "FY2010 change", 2015.5: "FY2016 change"}
ALL = lambda d: d[d.fy >= 2005]  # community colleges start in FY2005, so combined charts start there
tot = ex.groupby(["fy", "system"], as_index=False).amount_k.sum()
stacked_area_toggle(add_views(ALL(tot)), "system", "Spending by California's public higher education systems", HE + ". All spending, including hospitals, housing and research.", order=SYSTEMS).show()

## Who pays
Revenue by source, all three systems together. 'State appropriations' is the general-fund money the Legislature gives the systems; 'Local property taxes' is community colleges' share of local property tax; 'State grants' includes Cal Grants for students.

In [2]:
stacked_area_toggle(add_views(ALL(rev.groupby(["fy", "source"], as_index=False).amount_k.sum())), "source", "Revenue of public higher education, by source",
                    HE + ". All three systems.", order=REVENUE_ORDER).show()

### By system
The same revenue sources for each system. UC is mostly hospitals and research grants; CSU is state appropriations and tuition; community colleges are state money and local property taxes, with very low tuition.

In [3]:
for s in SYSTEMS:
    r = rev[rev.system == s]
    r = r[r.groupby("source").amount_k.transform("sum") > 0]
    stacked_area_toggle(add_views(r), "source", f"{s}: revenue by source", HE, order=[x for x in REVENUE_ORDER if x in set(r.source)]).show()

## Per student: public support versus tuition
Real (inflation-adjusted) dollars per full-time-equivalent student, in each system. Solid lines are public support (state appropriations and grants plus, for community colleges, local property taxes); dashed lines are tuition and fees paid. Per-student amounts are for tracking each system over time, not for comparing systems: UC's base includes research and hospital budgets.

In [4]:
f = fte.set_index(["fy", "system"]).fte
def per_fte(sources):
    x = rev[rev.source.isin(sources)].groupby(["fy", "system"], as_index=False).amount_k.sum()
    x = add_views(x)
    x["v"] = x.real_b * 1e9 / x.set_index(["fy", "system"]).index.map(f).values
    return x
pub = per_fte(["State appropriations", "State grants and contracts", "Local property taxes (community colleges)"])
tui = per_fte(["Tuition and fees"])
fig = go.Figure()
for i, s in enumerate(SYSTEMS):
    for x, nm, dash in ((pub, "public support", "solid"), (tui, "tuition and fees", "dash")):
        z = x[x.system == s]
        fig.add_trace(go.Scatter(x=z.fy, y=z.v, name=f"{s}: {nm}", mode="lines+markers", line=dict(color=PALETTE[i], dash=dash), marker=dict(size=4),
                                 hovertemplate="$%{y:,.0f}"))
fig.update_layout(yaxis_title=f"{base_fy()} dollars per FTE student", xaxis_title="Fiscal year (ending June 30)")
base_layout(fig, "Public support and tuition per student", HE + ". Real dollars per full-time-equivalent student; credit instruction only.")
fig.show()

## What the money is for
Spending by function. Instruction is teaching; research and public service are mostly UC; hospitals are UC only; institutional support is administration.

In [5]:
fa = ex.groupby(["fy", "function"], as_index=False).amount_k.sum()
stacked_area_toggle(add_views(ALL(fa)), "function", "Spending of public higher education, by function", HE + ". All three systems. Dotted lines: reporting changes.",
                    order=[x for x in FUNCTION_ORDER if x in set(fa.function)], breaks=BRK).show()

### By system

In [6]:
for s in SYSTEMS:
    z = ex[ex.system == s]
    stacked_area_toggle(add_views(z), "function", f"{s}: spending by function", HE + ". Dotted lines: reporting changes.",
                        order=[x for x in FUNCTION_ORDER if x in set(z.function)], breaks=BRK).show()

## Salaries, benefits and everything else
The same spending by natural class: salaries and wages, employee fringe benefits, and everything else (supplies, services, utilities, financial aid, depreciation, interest).

In [7]:
stacked_area_toggle(add_views(ALL(nat.groupby(["fy", "group"], as_index=False).amount_k.sum())), "group", "Salaries, benefits and other spending in public higher education",
                    HE + ". All three systems.", order=["Salaries and wages", "Fringe benefits", "Everything else"]).show()
t = nat.pivot_table(index=["system", "fy"], columns="group", values="amount_k", aggfunc="sum")
t["ratio"] = t["Fringe benefits"] / t["Salaries and wages"] * 100
fig = go.Figure()
for i, s in enumerate(SYSTEMS):
    z = t.loc[s]
    fig.add_trace(go.Scatter(x=z.index, y=z.ratio, name=s, mode="lines+markers", line=dict(color=PALETTE[i]), marker=dict(size=4), hovertemplate="%{y:.0f} cents"))
fig.update_layout(yaxis_title="Cents of benefits per dollar of salary", xaxis_title="Fiscal year (ending June 30)")
base_layout(fig, "Benefits per dollar of salary", HE + ". Includes pension and retiree-health expense as accounted (GASB 68/75), which swings year to year.")
fig.show()

## Spending on the core mission, per student
Spending on instruction, research, public service, academic support, student services, administration and plant operation (not hospitals, housing and dining, financial aid or independent operations), per full-time-equivalent student, in real dollars. Through FY2009 depreciation and interest are excluded from this total, which makes that period a few percent lower than later years.

In [8]:
core = ex[ex.function.isin(CORE)].groupby(["fy", "system"], as_index=False).amount_k.sum()
core = add_views(core)
core["v"] = core.real_b * 1e9 / core.set_index(["fy", "system"]).index.map(f).values
fig = go.Figure()
for i, s in enumerate(SYSTEMS):
    z = core[core.system == s]
    fig.add_trace(go.Scatter(x=z.fy, y=z.v, name=s, mode="lines+markers", line=dict(color=PALETTE[i]), marker=dict(size=4), hovertemplate="$%{y:,.0f}"))
fig.update_layout(yaxis_title=f"{base_fy()} dollars per FTE student", xaxis_title="Fiscal year (ending June 30)")
fig.add_vline(x=2009.5, line_dash="dot", line_color="grey", annotation_text="FY2010 change", annotation_position="bottom left")
base_layout(fig, "Core spending per student", HE + ". Real dollars per full-time-equivalent student.")
fig.show()

**What the data show** (real dollars are 2026 dollars; FY2005 to FY2023 unless noted)
- **Spending roughly doubled in real terms:** from $43.6B to $83.4B across the three systems. UC went from $24.3B to $54.2B, with hospitals and clinics rising from 25% to 34% of its spending; CSU from $7.6B to $11.8B; community colleges from $11.7B to $17.4B. Enrollment grew much less: UC full-time-equivalent students were up 44%, CSU 23%, and community colleges' credit FTE fell back to below 2005 levels after 2019.
- **State money fell in the recession and came back:** state appropriations to the three systems were $12.3B in 2005, $9.9B in 2012 and $15.6B in 2023. Per student, CSU went from $11.9k to $7.3k (2012) to $14.9k; UC from $21.3k to $13.4k to $16.1k; community colleges from $5.2k to $4.7k to $6.3k, before local property taxes (about a quarter of their revenue throughout).
- **Tuition filled part of the gap, unevenly:** per student, CSU tuition and fees rose from $5.6k to $8.4k (2012) and eased to $6.2k; UC's rose from $12.9k to $24.2k (2019) and $20.9k in 2023 (these figures include graduate and out-of-state tuition and are net of financial-aid discounts); community colleges stayed near $1.0k to $1.25k. The state's share of revenue went from 48% to 30% (2013) back to 45% at CSU, and from 16% to 9% at UC, as hospital and research revenue grew.
- **Core spending per student:** instruction, research, student services, administration and plant operation per FTE student, in real dollars, rose 11% at CSU ($17.9k to $19.8k), 27% at UC ($72k to $91k) and 37% at community colleges ($11.3k to $15.4k).
- **Benefits:** reported benefits per dollar of salary rose from 33 to 49 cents at CSU, 21 to 30 at UC and 28 to 37 at community colleges, with large swings in between (CSU peaked at 87 cents in 2018) that reflect pension and retiree-health accounting rather than cash.